# 1. Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler

### The below path should be where your data is saved (e.g. my data is savd in C:\Users\alexm\Documents\GitHub\SML_group_project\data")

In [ ]:
import os
os.chdir(r"C:\Users\alexm\Documents\GitHub\SML_group_project")
print("Working directory:", os.getcwd())

# 2. Load Data

In [ ]:
chargers = pd.read_csv("data/EV_charger_locations.csv")
ev_reg = pd.read_csv("data/EV_registration_activity.csv")
cities_pop = pd.read_csv("data/cities_counties_pop.csv")
income = pd.read_csv("data/median_income_by_county.csv")

print("chargers:", chargers.shape)
print("ev_reg:", ev_reg.shape)
print("cities_pop:", cities_pop.shape)
print("income:", income.shape)

# 3. Data Preparation #1: Data Cleaning

## 3.1 Clean EV registration data

In [ ]:
ev = ev_reg.copy()

# Standardize column names
ev.columns = (
    ev.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

In [ ]:
# Convert numeric columns
numeric_cols = [
    'model_year',
    'electric_range',
    'odometer_reading',
    'sale_price',
    'transaction_year'
]

for col in numeric_cols:
    if col in ev.columns:
        ev[col] = pd.to_numeric(ev[col], errors='coerce')

# Convert date columns
date_cols = [
    'sale_date',
    'transaction_date'
]

for col in date_cols:
    if col in ev.columns:
        ev[col] = pd.to_datetime(ev[col], errors='coerce')

In [ ]:
# Replace placeholder zeros with NaN for matrix completion later
ev['sale_price'] = ev['sale_price'].replace(0, np.nan)
ev['electric_range'] = ev['electric_range'].replace(0, np.nan)

In [ ]:
# Standardize county, city, and state text
for col in ['county', 'city', 'state']:
    if col in ev.columns:
        ev[col] = (
            ev[col]
            .replace(['nan', 'NaN', 'NAN', 'None', ''], np.nan)
            .astype(str)
            .str.strip()
            .str.upper()
        )

# Convert text "NAN" back to real missing values after uppercasing
ev['county'] = ev['county'].replace('NAN', np.nan)

# Create vehicle age
ev['vehicle_age'] = 2026 - ev['model_year']

# Drop EV rows with missing county because they cannot be used in county-level aggregation
ev = ev.dropna(subset=['county'])

ev.shape

In [ ]:
# Keep Washington records only
ev = ev[ev['state'] == 'WA'].copy()

# Drop rows without county
ev['county'] = ev['county'].replace(['NAN', 'nan', 'None', ''], np.nan)
ev = ev.dropna(subset=['county'])

## 3.2 Matrix completion note for `electric_range`

Placeholder zero values in `electric_range` were converted to missing values above. Matrix completion is applied later at the county level after aggregation. This avoids over smoothing millions of vehicle level rows.

## 3.3 Clean charger location data

In [ ]:
ch = chargers.copy()

# Standardize column names
ch.columns = (
    ch.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

In [ ]:
# Convert date columns
date_cols = [
    'date_last_',
    'date_las_1',
    'open_date',
    'availabili'
]

for col in date_cols:
    if col in ch.columns:
        ch[col] = pd.to_datetime(ch[col], errors='coerce')

# Standardize city and state text
for col in ['city', 'state']:
    if col in ch.columns:
        ch[col] = (
            ch[col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

In [ ]:
# Remove irrelevant text/ID fields for county-level analysis
drop_cols = [
    'x', 'y', 'fid', 'objectid', 'station_na', 'station_id',
    'address', 'directions', 'phone_numb',
    'ev_netwo_2', 'access_det', 'ev_pricing'
]

ch = ch.drop(columns=drop_cols, errors='ignore')

# Confirm duplicate and charger-count status
print("Duplicate charger rows:", ch.duplicated().sum())

charger_count_cols = ['ev_level1_', 'ev_level2_', 'ev_dc_fast', 'total_leve']
ch[charger_count_cols].isna().sum()

## 3.4 Clean city/county lookup data

In [ ]:
cp = cities_pop.copy()

# Standardize column names
cp.columns = (
    cp.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Standardize city and county text
cp['city'] = (
    cp['city']
    .astype(str)
    .str.strip()
    .str.upper()
)

cp['county'] = (
    cp['county']
    .astype(str)
    .str.strip()
    .str.upper()
)

cp.head()

## 3.5 Clean income data

In [ ]:
income_2025 = income[['County', '2025']].copy()

income_2025.columns = ['county', 'median_income_2025']

income_2025['county'] = (
    income_2025['county']
    .str.upper()
    .str.strip()
)

income_2025['median_income_2025'] = pd.to_numeric(
    income_2025['median_income_2025'],
    errors='coerce'
)

income_2025.head()

# 4. Data Preparation #2: Add County to Charger Data

In [ ]:
# Initial city to county merge
city_county = cp[['city', 'county']].drop_duplicates()

ch = ch.merge(
    city_county,
    on='city',
    how='left'
)

print("Missing charger county mappings after city merge:", ch['county'].isna().sum())

In [ ]:
# Manual fixes for charger cities not matched by the city/county lookup
manual_county_map = {
    'VANCOUVER': 'CLARK',
    'WALLA WALLA': 'WALLA WALLA',
    'TUKWILA': 'KING',
    'YAKIMA': 'YAKIMA',
    'WOODINVILLE': 'KING',
    'WENATCHEE': 'CHELAN',
    'WASHOUGAL': 'CLARK',
    'SILVERDALE': 'KITSAP',
    'TUMWATER': 'THURSTON',
    'MT. VERNON': 'SKAGIT',
    'MT VERNON': 'SKAGIT',
    'EASTSOUND': 'SAN JUAN',
    'WAPATO': 'YAKIMA',
    'KINGSTON': 'KITSAP',
    'WINTHROP': 'OKANOGAN',
    'MANSON': 'CHELAN',
    'NEAH BAY': 'CLALLAM',
    'WHITE SALMON': 'KLICKITAT',
    'ZILLAH': 'YAKIMA',
    'SNOQUALMIE PASS': 'KING',
    'LOPEZ': 'SAN JUAN',
    'LOPEZ ISLAND': 'SAN JUAN',
    'ASHFORD': 'PIERCE',
    'ROCKPORT': 'SKAGIT',
    'SUQUAMISH': 'KITSAP',
    'AURORA': 'KING',
    'MOCLIPS': 'GRAYS HARBOR',
    'BELFAIR': 'MASON',
    'TOPPENISH': 'YAKIMA',
    'ELBE': 'PIERCE',
    'SALKUM': 'LEWIS',
    'QUINAULT': 'GRAYS HARBOR',
    'CAMP MURRAY': 'PIERCE',
    'TULALIP': 'SNOHOMISH',
    'CAMANO': 'ISLAND',
    'WEST SEATTLE': 'KING',
    'VASHON': 'KING',
    'MEAD': 'SPOKANE',
    'WATERVILLE': 'DOUGLAS',
    'ROCHE HARBOR': 'SAN JUAN',
    'GRAYLAND': 'GRAYS HARBOR',
    'WAITSBURG': 'WALLA WALLA',
    'YELM': 'THURSTON',
    'MAZAMA': 'OKANOGAN',
    'COUGAR': 'COWLITZ',
    'CUSTER': 'WHATCOM',
    'TWISP': 'OKANOGAN',
    'TONASKET': 'OKANOGAN',
    'OLGA': 'SAN JUAN',
    'COPALIS CROSSING': 'GRAYS HARBOR',
    'ROCHESTER': 'THURSTON',
    'CARSON': 'SKAMANIA',
    'FREELAND': 'ISLAND',
    'SEAVIEW': 'PACIFIC',
    'UNION': 'MASON',
    'WESTPORT': 'GRAYS HARBOR',
    'FALL CITY': 'KING',
    'RICHLAND': 'BENTON',
    'RICHALND': 'BENTON',
    'BURIEN': 'KING',
    'LBURIEN': 'KING',
    'LYNNWOOD': 'SNOHOMISH',
    'LYNWOOD': 'SNOHOMISH',
    'SEA TAC': 'KING',
    'QUIL CEDA VILLAGE': 'SNOHOMISH',
    'PACIFIC BEACH': 'GRAYS HARBOR',
    'VISTA DR': 'SPOKANE',
    'ST': 'PIERCE'
}

ch['county'] = ch['county'].fillna(ch['city'].map(manual_county_map))

# Resolve ambiguous split-county city labels in charger data
ch.loc[ch['city'] == 'BOTHELL', 'county'] = 'KING'
ch.loc[ch['city'] == 'AUBURN', 'county'] = 'KING'

print("Missing charger county mappings after manual fixes:", ch['county'].isna().sum())

# 5. Data Preparation #3: County-Level Aggregation

## 5.1 Aggregate EV adoption features by county

In [ ]:
# Aggregate EV adoption features by county
ev_county = (
    ev.groupby('county')
    .agg(
        total_EV_count=('county', 'size'),
        BEV_count=(
            'clean_alternative_fuel_vehicle_type',
            lambda x: (x == 'Battery Electric Vehicle (BEV)').sum()
        ),
        PHEV_count=(
            'clean_alternative_fuel_vehicle_type',
            lambda x: (x == 'Plug-in Hybrid Electric Vehicle (PHEV)').sum()
        ),
        avg_model_year=('model_year', 'mean'),
        avg_electric_range=('electric_range', 'mean'),
        avg_vehicle_age=('vehicle_age', 'mean')
    )
    .reset_index()
)

# Derived EV features
ev_county['percent_BEV'] = ev_county['BEV_count'] / ev_county['total_EV_count']
ev_county['percent_PHEV'] = ev_county['PHEV_count'] / ev_county['total_EV_count']

# Avoid divide by zero
ev_county['BEV_to_PHEV_ratio'] = (
    ev_county['BEV_count'] /
    ev_county['PHEV_count'].replace(0, np.nan)
)

ev_county.head()

## 5.2 County-level matrix completion for `avg_electric_range`

This performs the matrix-completion step, but at the county level rather than the individual vehicle level. The observed county averages are kept fixed, and only missing county level `avg_electric_range` values are completed. If no counties have missing averages, the column is left unchanged, which is what happens in this final version.

In [ ]:
# Matrix completion at county level for avg_electric_range
range_impute_cols = [
    'total_EV_count',
    'BEV_count',
    'PHEV_count',
    'percent_BEV',
    'percent_PHEV',
    'avg_model_year',
    'avg_vehicle_age',
    'avg_electric_range',
    'BEV_to_PHEV_ratio'
]

# Keep a copy of the observed county averages for transparency
ev_county['avg_electric_range_observed'] = ev_county['avg_electric_range']

missing_avg_range = ev_county['avg_electric_range'].isna()
print('Missing county avg_electric_range before matrix completion:', missing_avg_range.sum())

if missing_avg_range.sum() > 0:
    range_matrix = ev_county[range_impute_cols].replace([np.inf, -np.inf], np.nan).copy()

    scaler = StandardScaler()
    range_scaled = scaler.fit_transform(range_matrix)

    imputer = IterativeImputer(
        max_iter=20,
        random_state=42,
        initial_strategy='median'
    )

    range_completed_scaled = imputer.fit_transform(range_scaled)

    range_completed = pd.DataFrame(
        scaler.inverse_transform(range_completed_scaled),
        columns=range_impute_cols,
        index=ev_county.index
    )

    ev_county.loc[missing_avg_range, 'avg_electric_range'] = (
        range_completed.loc[missing_avg_range, 'avg_electric_range']
    )

# Electric range cannot be negative
ev_county['avg_electric_range'] = ev_county['avg_electric_range'].clip(lower=0)

print('Missing county avg_electric_range after matrix completion:', ev_county['avg_electric_range'].isna().sum())

ev_county[['county', 'avg_electric_range_observed', 'avg_electric_range']].head()

Electric range contained missing values after placeholder zeros were converted to NaN. Matrix completion was tested, but around 44% of vehicle records required imputation. At the vehicle level, this produced overly similar electric range values and reduced natural variation in the data suggesting over-smoothing.

Because county level aggregation used the mean of observed values (mean() ignores NaNs), all counties still retained valid average electric range estimates and no county level missing values remained.

Since the goal of this project is county level PCA and clustering, preserving observed county level patterns was preferred over imputing a large number of vehicle level values. County averages were therefore calculated from observed data, with matrix completion evaluated but not actually needed in the final dataset.

## 5.3 Merge county-level income

In [ ]:
ev_county = ev_county.merge(
    income_2025,
    on='county',
    how='left'
)

print("Missing income values:", ev_county['median_income_2025'].isna().sum())
ev_county.head()

## 5.4 Population feature note

Attempted to make `EV_per_1000_people`, but the `cities_counties_pop` file appears to contain city level populations rather than complete county populations. Summing those city values produced impossible EV-per-1000 values, so `Population_2025` and `EV_per_1000_people` were excluded from the final cleaned dataset. `Population_density` was also not created because we don't have land area data. We would need another dataset for this.

## 5.5 Aggregate charger features by county

In [ ]:
charger_county = (
    ch.groupby('county')
    .agg(
        total_chargers=('total_leve', 'sum'),
        level1_chargers=('ev_level1_', 'sum'),
        level2_chargers=('ev_level2_', 'sum'),
        dc_fast_chargers=('ev_dc_fast', 'sum')
    )
    .reset_index()
)

charger_county['fast_charger_share'] = (
    charger_county['dc_fast_chargers'] /
    charger_county['total_chargers']
)

charger_county.head()

# 6. Final County-Level Dataset

In [ ]:
# Keep the final feature set concise
ev_county_final = ev_county.drop(columns=['avg_electric_range_observed'], errors='ignore')

county_data = ev_county_final.merge(
    charger_county,
    on='county',
    how='left'
)

# Counties with no charger records should have charger counts of 0
charger_cols = [
    'total_chargers',
    'level1_chargers',
    'level2_chargers',
    'dc_fast_chargers',
    'fast_charger_share'
]

county_data[charger_cols] = county_data[charger_cols].fillna(0)

county_data['chargers_per_1000_EV'] = (
    county_data['total_chargers'] /
    county_data['total_EV_count']
) * 1000

county_data.head()

# 7. Final Validation

In [ ]:
print("Final dataset shape:", county_data.shape)
print("Missing values:")
print(county_data.isna().sum())

county_data.describe()

# 8. Save Cleaned Data

In [ ]:
county_data.to_csv("data/clean_county_data.csv", index=False)
ch.to_csv("data/clean_chargers.csv", index=False)
charger_county.to_csv("data/charger_county.csv", index=False)
ev_county.to_csv("data/ev_county.csv", index=False)

print("Saved cleaned datasets.")